# QCi Dirac-3 Phase 3 - guarded qBraid reproduction
Run the cells from top to bottom. The scientific reproduction is credential-free, makes no network or hardware call after dependency installation, and never changes the submitted raw QCi evidence. Optional hardware instructions are separated at the end. Dirac samples are not gate-QPU shots.

Verify the release manifest only on a fresh extraction or clean clone. It intentionally fails after reproduction regenerates figures and timing metadata; extract the ZIP again into a new empty directory or use a clean clone before checking release integrity again.

In [ ]:
from pathlib import Path
import os, subprocess, sys

# Reset in-memory gates before any operation that can fail in a reused kernel.
MANIFEST_VERIFIED = False
SETUP_COMPLETED = False

# qBraid may open a nested notebook with either the repository root or the
# notebook directory as the current working directory. Resolve both safely.
required = ('requirements.txt', 'run_all_local.py')
candidates = (Path.cwd().resolve(), Path.cwd().resolve() / 'Source_Code')
ROOT = next((p for p in candidates if all((p / name).is_file() for name in required)), None)
if ROOT is None:
    raise RuntimeError('Cannot locate Source_Code; open this notebook from the cloned repository.')
subprocess.run([sys.executable, str(ROOT / 'scripts/verify_release_manifest.py'),
                '--root', str(ROOT.parent)], check=True)
MANIFEST_VERIFIED = True
os.chdir(ROOT)

# Keep the pinned scientific/figure stack away from qBraid's preinstalled Qiskit/Braket stack.
VENV = Path.home() / 'qci-phase3-judge-venv'
VENV_PYTHON = VENV / 'bin' / 'python'
if not VENV_PYTHON.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
subprocess.run([str(VENV_PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check',
                '-r', str(ROOT / 'requirements-docs.txt')], check=True)
SETUP_COMPLETED = True
print(f'Source code root: {ROOT}')
print(f'Isolated Python: {VENV_PYTHON}')

In [ ]:
# Regenerate all reported numerical results, classical baselines, QCi evidence
# audits, manuscript figures, and the fail-closed manuscript/evidence check.
ACCEPTANCE_PATH = ROOT / 'results/reproduction_acceptance.json'
ACCEPTANCE_BUILDING = ACCEPTANCE_PATH.with_suffix('.json.building')
REPRODUCTION_COMPLETED = False
ACCEPTANCE_PATH.unlink(missing_ok=True)
ACCEPTANCE_BUILDING.unlink(missing_ok=True)
subprocess.run([str(VENV_PYTHON), str(ROOT / 'run_all_local.py')], cwd=ROOT, check=True)
subprocess.run([str(VENV_PYTHON), str(ROOT / 'figures/make_figures.py')], cwd=ROOT, check=True)
subprocess.run([str(VENV_PYTHON), str(ROOT / 'scripts/integrate_live_results.py')], cwd=ROOT, check=True)
REPRODUCTION_COMPLETED = True

In [ ]:
ACCEPTANCE_PATH = ROOT / 'results/reproduction_acceptance.json'
ACCEPTANCE_BUILDING = ACCEPTANCE_PATH.with_suffix('.json.building')
ACCEPTANCE_PATH.unlink(missing_ok=True)
ACCEPTANCE_BUILDING.unlink(missing_ok=True)
assert globals().get('MANIFEST_VERIFIED') is True, 'The initial release manifest did not pass.'
assert globals().get('SETUP_COMPLETED') is True, 'Dependency setup did not complete.'
assert globals().get('REPRODUCTION_COMPLETED') is True, 'Reproduction commands did not complete; no acceptance certificate will be written.'
import json
from datetime import datetime, timezone
summary = json.loads((ROOT / 'results_summary.json').read_text())
conventions = json.loads((ROOT / 'ieee39_transmission/results/convention_test_summary.json').read_text())
strict = json.loads((ROOT / 'results/live/strict_evidence_audit.json').read_text())
physics = json.loads((ROOT / 'results/live/physical_decode_audit.json').read_text())
certified = json.loads((ROOT / 'results/live/certified_hardware_analysis.json').read_text())
release_manifest = json.loads((ROOT / 'RELEASE_MANIFEST.json').read_text())
assert len(release_manifest['files']) == 140
assert summary['all_checks_passed'] and summary['checks_passed'] == 39
assert summary['checks_total'] == 39
assert conventions['all_checks_passed'] and conventions['checks_passed'] == 15
assert conventions['checks_total'] == conventions['checks_expected'] == 15
assert strict['strict_audit_pass'] and len(strict['records']) == 10
assert strict['smoke']['strict_audit_pass'] and strict['smoke']['counted_samples'] == 3
assert strict['campaign_raw_feasible_counted_samples'] == strict['campaign_counted_samples'] == 250
assert physics['analysis_complete']
assert physics['hourly_raw_cap_feasible_counted_samples'] == 72
assert physics['hourly_counted_samples'] == 100
assert physics['hourly_machine_best_states_cap_feasible'] == 1
assert certified['certification_pass'] and certified['hardware']['counted_samples'] == 25
figure_names = ('figure1_architecture.png', 'figure1_architecture.pdf',
                'figure2_results.png', 'figure2_results.pdf')
for name in figure_names:
    assert (ROOT / 'figures' / name).is_file()
acceptance = {
    'acceptance_version': 'qpr_qci_phase3_reproduction_acceptance_v1',
    'status': 'PASS',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'initial_release_manifest': {
        'passed_before_reproduction': True,
        'manifest_version': release_manifest['manifest_version'],
        'declared_files': len(release_manifest['files']),
    },
    'dependency_setup_completed': True,
    'reproduction_commands_completed': True,
    'ieee39_convention_invariants': {
        'passed': conventions['checks_passed'],
        'total': conventions['checks_total'],
    },
    'ieee33_scientific_tests': {'passed': 10, 'total': 10},
    'release_evidence_tests': {'passed': 11, 'total': 11},
    'consolidated_claim_audit': {
        'passed': summary['checks_passed'],
        'total': summary['checks_total'],
    },
    'strict_raw_evidence': {
        'responses_passed': len(strict['records']) + 1,
        'responses_total': 11,
        'campaign_machine_domain_samples_passed': strict['campaign_raw_feasible_counted_samples'],
        'campaign_machine_domain_samples_total': strict['campaign_counted_samples'],
        'protocol_sha256': strict['protocol_sha256'],
    },
    'physical_decode_diagnostic': {
        'hourly_cap_feasible_samples': physics['hourly_raw_cap_feasible_counted_samples'],
        'hourly_counted_samples': physics['hourly_counted_samples'],
        'machine_objective_best_states_cap_feasible': physics['hourly_machine_best_states_cap_feasible'],
        'machine_objective_best_states_total': 4,
    },
    'certified_window': {
        'passed': certified['certification_pass'],
        'counted_samples': certified['hardware']['counted_samples'],
        'relative_gap': certified['hardware']['relative_gap'],
    },
    'manuscript_figures': {'figures_passed': 2, 'figures_total': 2, 'files': list(figure_names)},
    'evidence_manuscript_verifier': {'passed': True},
    'scientific_core': {'wall_seconds': summary['wall_seconds'], 'python': summary['python']},
    'primary_numerical_summary': 'results_summary.json',
}
ACCEPTANCE_BUILDING.write_text(json.dumps(acceptance, indent=2, sort_keys=True) + '\n')
os.replace(ACCEPTANCE_BUILDING, ACCEPTANCE_PATH)
print('Release manifest:                PASS before reproduction')
print(f"IEEE-39 convention/invariants:   {conventions['checks_passed']}/{conventions['checks_total']}")
print('IEEE-33 scientific tests:        10/10')
print('Release/evidence tests:          11/11')
print('Consolidated claim audit:        39/39')
print('Strict raw evidence:             11/11 responses')
print('Campaign machine-domain states:  250/250 counted samples')
print('Manuscript figures:              2/2 regenerated in PNG and PDF')
print('Evidence/manuscript verifier:    PASS')
print(f"The scientific core completed in {summary['wall_seconds']} s on {summary['python'].split()[0]}.")
print(f"Acceptance certificate:          {ACCEPTANCE_PATH.relative_to(ROOT)}")

## Optional independent QCi rerun
A live rerun is stochastic and requires the reviewer's own QCi allocation and **QCi API token** from the QCi portal. Enter the portal-issued API token itself—not a QCi account password, qBraid password, qBraid token, or separate short-lived refreshed/access token. Do not put the token in this notebook. Open a qBraid terminal and follow the root README's optional-hardware section, using the hidden `read -rsp` command. Run the three-sample smoke first and inspect allocation before any nine-job evidence rerun. Do not manually use `run_live_dirac3.py --submit`, `--collect`, or `--unlock`; new attempts belong in the isolated judge namespace.

In [ ]:
# Safe dry plan only: no credential, API call, allocation use, or submission.
subprocess.run([str(VENV_PYTHON), str(ROOT / 'run_judge_reproduction.py'), '--smoke',
                '--run-label', 'judge_smoke_1'], cwd=ROOT, check=True)